In [1]:
import requests
import re
import time
from pathlib import Path
from typing import List, Dict
from urllib.parse import urljoin
from dataclasses import dataclass
from bs4 import BeautifulSoup
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm import tqdm
from firecrawl import FirecrawlApp
from crawl4ai import AsyncWebCrawler

In [2]:
# Configure settings in terms of 
## Google Sheet 
## Batch Size 
## Credentials File Path
## Column to read URL from
## Column to write the result into
## What Content is this (About US, Ebook, etc...)

In [3]:
GOOGLE_SHEET_LINK = "https://docs.google.com/spreadsheets/d/1QtKOB5ChRemg2_wJOxeZhn1qao1ZrRjVK8nExVzbxEI/edit?gid=2011509251#gid=2011509251"
BATCH_SIZE = 50
CREDENTIALS_FILE_PATH = "/Users/utkarshumang/my_projects/lead-enricher-ai-be/data/url-to-email-445616-cebe4868914f.json"
COLUMN_TO_READ_URL_FROM = "G"
COLUMN_TO_WRITE_URL_TO = {
	"ABOUT_US": "M",
	"EBOOK": "N",
	"COURSES": "O",
	"RECENT_BLOG": "P",
	"TESTIMONIALS": "Q",
	"WEBINAR": "R",
	"SERVICES": "S",
	"PODCAST": "T",
	"SHOP": "U"
}
FIRECRAWL_API_KEY = "fc-29599096ac8b426dbf178180c53500ed"

In [4]:
# Enter which column to process here
COLUMN_TO_PROCESS = "RECENT_BLOG" 
COLUMN_KEYWORDS = {
	"EBOOK": ["ebook", "e-book", "downloads", "whitepaper", "guide", "brochure"],
	"ABOUT_US": ["about", "who-we-are", "company", "our-story", "mission", "vision"],
	"COURSES": ["course", "training", "academy", "learning", "bootcamp"],
	"RECENT_BLOG": ["blog", "insights", "articles", "news", "stories"],
	"TESTIMONIALS": ["testimonial", "reviews", "feedback", "case-studies", "customers"],
	"WEBINAR": ["webinar", "events", "live", "sessions", "recording"],
	"SERVICES": ["service", "solutions", "offerings", "capabilities"],
	"PODCAST": ["podcast", "episodes", "listen", "audio"],
	"SHOP": ["shop", "store", "buy", "product", "checkout", "cart"]
}

In [5]:
@dataclass
class Config:
	"""Configuration settings for scraping."""
	max_retries: int = 3
	request_timeout: int = 30
	delay_between_requests: float = 2.0
	max_content_length: int = 10000

# Initialize config
config = Config()

In [6]:
class GoogleSheetsManager:
	"""Manages interactions with Google Sheets."""
	
	def __init__(self, credentials_file: str):
		self.credentials_file = credentials_file
		self.service = self._setup_service()
	
	def _setup_service(self):
		"""Initialize Google Sheets API service."""
		if not Path(self.credentials_file).exists():
			raise FileNotFoundError(f"Credentials file not found: {self.credentials_file}")
		
		scopes = ['https://www.googleapis.com/auth/spreadsheets']
		creds = service_account.Credentials.from_service_account_file(
			self.credentials_file, scopes=scopes
		)
		return build('sheets', 'v4', credentials=creds)
	
	def extract_spreadsheet_id(self, sheet_url: str) -> str:
		"""Extract spreadsheet ID from Google Sheets URL."""
		pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
		match = re.search(pattern, sheet_url)
		if match:
			return match.group(1)
		raise ValueError(f"Invalid Google Sheet URL: {sheet_url}")
	
	def get_urls(self, spreadsheet_id: str) -> List[str]:
		"""Retrieve URLs from Google Sheet dynamically."""
		range_name = f"{COLUMN_TO_READ_URL_FROM}:{COLUMN_TO_READ_URL_FROM}"
		try:
			result = self.service.spreadsheets().values().get(
				spreadsheetId=spreadsheet_id,
				range=range_name
			).execute()
			values = result.get('values', [])
			urls = [row[0] for row in values[1:] if row and row[0].strip()]
			return urls
		except Exception as e:
			print(f"❌ Error fetching URLs: {e}")
			return []
	
	def update_results(self, spreadsheet_id: str, results: List[Dict]):
		"""Update Google Sheet with scraping results, using dynamic column based on COLUMN_TO_PROCESS."""
		if not results:
			return

		column_letter = COLUMN_TO_WRITE_URL_TO.get(COLUMN_TO_PROCESS)
		if not column_letter:
			print(f"❌ Invalid COLUMN_TO_PROCESS: {COLUMN_TO_PROCESS}")
			return

		max_retries = config.max_retries
		for attempt in range(max_retries):
			try:
				content_data = [[result['content']] for result in results]
				# You can include status column logic here if needed

				self.service.spreadsheets().values().update(
					spreadsheetId=spreadsheet_id,
					range=f"{column_letter}2:{column_letter}{len(content_data) + 1}",
					valueInputOption='RAW',
					body={'values': content_data}
				).execute()

				print(f"✅ Updated {len(results)} rows in column {column_letter}")
				return

			except HttpError as e:
				print(f"❌ HTTP Error on attempt {attempt + 1}: {e}")
				if e.resp.status == 429:
					print("Rate limit exceeded.")
			except Exception as e:
				print(f"❌ Error on attempt {attempt + 1}: {e}")

			if attempt < max_retries - 1:
				wait_time = 5 * (attempt + 1)
				print(f"⏳ Retrying in {wait_time} seconds...")
				time.sleep(wait_time)
			else:
				print("❌ All retries failed. Could not update sheet.")

In [7]:
app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)

map_result = app.map_url("https://www.marcusmillichap.com/")

In [8]:
print(map_result)
MAP_URLS = map_result.links

success=True links=['https://www.marcusmillichap.com', 'https://www.marcusmillichap.com/search', 'https://www.marcusmillichap.com/canada', 'https://www.marcusmillichap.com/properties', 'https://www.marcusmillichap.com/careers', 'https://www.marcusmillichap.com/research', 'https://www.marcusmillichap.com/auctions', 'https://www.marcusmillichap.com/zangcrotts', 'https://www.marcusmillichap.com/services', 'https://www.marcusmillichap.com/contact-us', 'https://www.marcusmillichap.com/stewart-group', 'https://www.marcusmillichap.com/heitzeberg-group', 'https://www.marcusmillichap.com/about-us', 'https://www.marcusmillichap.com/castellanos-team', 'https://www.marcusmillichap.com/whiteside-group', 'https://www.marcusmillichap.com/attia-group', 'https://www.marcusmillichap.com/laaa-team', 'https://www.marcusmillichap.com/properties/documents', 'https://www.marcusmillichap.com/advisors/joel-westle', 'https://www.marcusmillichap.com/advisors/dylan-hellberg', 'https://www.marcusmillichap.com/advi

In [9]:
# Filter for specific suburls which might be related to the column being processed
def filter_urls_by_column(urls: list, column: str) -> list:
	keywords = COLUMN_KEYWORDS.get(column.upper(), [])
	if not keywords:
		print(f"⚠️ No keywords defined for column: {column}")
		return []

	filtered = [
		url for url in urls
		if any(keyword in url.lower() for keyword in keywords)
	]
	return filtered

In [10]:
filtered_url = filter_urls_by_column(MAP_URLS, COLUMN_TO_PROCESS)

In [11]:
class Crawl4AIExtractor:
	def __init__(self, urls: List[str]):
		self.urls = urls

	async def extract_first_url(self) -> Dict:
		if not self.urls:
			return {"status": "fail", "reason": "No URLs provided."}
		
		url = self.urls[0]
		try:
			async with AsyncWebCrawler() as crawler:
				result = await crawler.arun(url=url)
				return {
					"url": url,
					"status": "success",
					"content": result.html,
					"title": getattr(result, "title", None),
					"url_type": getattr(result, "url_type", None)
				}
		except Exception as e:
			return {
				"url": url,
				"status": "fail",
				"error": str(e)
			}

In [12]:
if len(filtered_url) > 0:
  extractor = Crawl4AIExtractor(filtered_url)
  result = await extractor.extract_first_url()

  print(result)
else:
  print("Nothing to Process here")

[INIT].... → Crawl4AI 0.6.3 

[FETCH]... ↓ https://www.marcusmillichap.com/news-events/press/2025/05/05-28-americinn                            |
✓ | ⏱: 2.88s 

[SCRAPE].. ◆ https://www.marcusmillichap.com/news-events/press/2025/05/05-28-americinn                            |
✓ | ⏱: 0.01s 

[COMPLETE] ● https://www.marcusmillichap.com/news-events/press/2025/05/05-28-americinn                            |
✓ | ⏱: 2.90s 

{'url': 'https://www.marcusmillichap.com/news-events/press/2025/05/05-28-americinn', 'status': 'success', 'content': '<!DOCTYPE html><html lang="en" style="" class=" js flexbox canvas canvastext webgl no-touch geolocation postmessage no-websqldatabase indexeddb hashchange history draganddrop websockets rgba hsla multiplebgs backgroundsize borderimage borderradius boxshadow textshadow opacity cssanimations csscolumns cssgradients cssreflections csstransforms csstransforms3d csstransitions fontface generatedcontent video audio localstorage sessionstorage webworkers no-applicationcache svg inlinesvg smil svgclippaths"><head>\n\n    <link rel="apple-touch-icon" sizes="180x180" href="/Areas/MM/img/favicon/apple-touch-icon.png">\n    <link rel="icon" type="image/png" sizes="32x32" href="/Areas/MM/img/favicon/favicon-32x32.png">\n    <link rel="icon" type="image/png" sizes="16x16" href="/Areas/MM/img/favicon/favicon-16x16.png">\n    <link rel="manifest" href="/Areas/MM/img/favicon/site.webman

In [13]:
from bs4 import BeautifulSoup

def extract_main_html_content(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    # Try main content tags first
    main = soup.find("main")
    if main:
        return main.get_text(separator="\n", strip=True)

    article = soup.find("article")
    if article:
        return article.get_text(separator="\n", strip=True)

    # Fallback: find largest <div> not known to be junk
    candidates = []
    for div in soup.find_all("div"):
        class_names = " ".join(div.get("class", []))
        if any(x in class_names.lower() for x in ["nav", "footer", "header", "cookie", "modal", "popup"]):
            continue
        text_len = len(div.get_text(strip=True))
        if text_len > 200:  # heuristic: only large divs
            candidates.append((text_len, div))

    if candidates:
        # Return text from the largest good div
        return max(candidates, key=lambda x: x[0])[1].get_text(separator="\n", strip=True)

    # Last resort: return full page text (minus scripts/styles)
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return soup.get_text(separator="\n", strip=True)

In [14]:
# Use this if you use result.html instead of markdown
html = result.get("content")
main_text = extract_main_html_content(html)

print(main_text)

Press Releases
Marcus & Millichap Facilitates Sale of 42-Room AmericInn Hotel in Rice Lake, Wisconsin
May 28, 2025
LinkedIn
Email App
RICE LAKE, Wis
., May 28, 2025 – Marcus & Millichap (NYSE: MMI), a leading commercial real estate brokerage firm specializing in investment sales, financing, research and advisory services, announced today the sale of a 42-room AmericInn hotel property in Rice Lake, Wisconsin.
“This transaction stood out due to the buyer’s close proximity to the property, which enables a more hands-on ownership style,” said Jon Ruzicka, first vice president investments. “That level of involvement is expected to strengthen daily operations and deepen guest relationships. With modest updates planned, the hotel will continue to offer the clean, comfortable experience travelers value. The Rice Lake hospitality market remains strong, with steady tourism and ongoing demand for well-run, midscale accommodations.”
Ruzicka, Joseph Ferguson and Reed Gizinski, investment specialist

In [15]:
# Write these back to google sheets and create a master function 
